# Main experiment analysis

Analysis code for Bui Ngoc Minh Quach’s 2026 master’s thesis at the Technical University of Munich. See the repository README for setup, data access and citation.

# 0. Setup and Environment

This chapter prepares the analysis environment by importing the required libraries, defining global settings and project folders, and loading the raw Qualtrics dataset. The data are not modified at this stage.


## 0.1 Required packages

Install the Python dependencies from the repository’s `requirements.txt` before starting JupyterLab.

## 0.2 Import Libraries

All package imports used throughout the notebook are consolidated below.


In [ ]:
import os
import sys
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency, levene, shapiro, ttest_ind

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

import pingouin as pg
import matplotlib.pyplot as plt

# Reproducibility
RANDOM_SEED = 2026
np.random.seed(RANDOM_SEED)

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

warnings.filterwarnings("ignore")

print("✓ Libraries successfully imported.")


## 0.3 Project Structure

The following folders keep the raw data, exported tables, and diagnostic figures in a consistent project structure.


In [ ]:
# Locate the repository from its root or the notebooks folder.
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "requirements.txt").is_file():
    raise RuntimeError("Start JupyterLab from the repository root.")

DATA_DIR = PROJECT_DIR / "data" / "main"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "main"
TABLE_DIR = OUTPUT_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)


## 0.4 Load the Raw Dataset

The raw Qualtrics export is loaded from `data/Final_Data.csv`. The file is read without modification.


In [ ]:
DATA_FILE = DATA_DIR / "Final_Data.csv"
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        "This notebook requires an authorized local copy of "
        "data/main/Final_Data.csv. See the repository README."
    )

df_raw = pd.read_csv(DATA_FILE)
print(f"Loaded {df_raw.shape[0]} rows and {df_raw.shape[1]} columns.")


## 0.5 Chapter Summary

The analysis environment and project folders are configured, and the raw Qualtrics dataset is loaded without modification. Data preparation begins in the next chapter.


# 1. Data Preparation

This chapter prepares the raw Qualtrics export for analysis. It inspects the file structure, removes export metadata, screens responses, applies the predefined exclusion criteria, and creates the final analytical sample while leaving the raw file unchanged.


## 1.1 Inspect the Raw Qualtrics Export

The raw export is inspected to confirm its dimensions, variables, and formatting before preprocessing. No data are changed in this step.


In [ ]:
print(f"Qualtrics export contains {len(df_raw)} rows, including two metadata rows.")

## 1.2 Remove Qualtrics Metadata

The Qualtrics export contains two metadata rows before the participant responses: question labels and Qualtrics import identifiers. Because these rows are not participant data, they are removed before further preprocessing.


In [ ]:
df_participants = df_raw.iloc[2:].copy().reset_index(drop=True)

## 1.3 Response Screening

Responses are screened for survey completion, availability of the experimental condition, the instructed-response attention check, and the hidden honeypot field. This section identifies potentially invalid responses; exclusions are applied in the next section.


In [ ]:
# Screen Participant Responses

# Key screening variables
finished_var = "Finished"
condition_var = "Condition"
attention_var = "Control_annoyance _4"
honeypot_var = "Honeypot"

# Convert relevant variables to numeric
df_participants[finished_var] = pd.to_numeric(
    df_participants[finished_var], errors="coerce"
)

df_participants[attention_var] = pd.to_numeric(
    df_participants[attention_var], errors="coerce"
)

# Survey completion

n_complete = (df_participants[finished_var] == 1).sum()
n_incomplete = (df_participants[finished_var] != 1).sum()

# Experimental condition

condition_counts = df_participants[condition_var].value_counts(dropna=False)

n_no_game = condition_counts.get("No_Game", 0)
n_game_certain = condition_counts.get("Game_Certain", 0)
n_game_cb = condition_counts.get("Game_CB", 0)
n_condition_missing = df_participants[condition_var].isna().sum()

# ------------------------------------------------------------
# Attention check
# Correct response: 6 = "Agree"
# ------------------------------------------------------------

n_attention_pass = (df_participants[attention_var] == 6).sum()

n_attention_fail = (
    df_participants[attention_var].notna()
    & (df_participants[attention_var] != 6)
).sum()

n_attention_missing = df_participants[attention_var].isna().sum()

# Honeypot

n_honeypot = df_participants[honeypot_var].notna().sum()

# Screening overview

print("RESPONSE SCREENING")
print("-" * 50)

print(f"Participant responses : {len(df_participants)}")
print(f"Completed surveys     : {n_complete}")
print(f"Incomplete surveys    : {n_incomplete}")

print("\nExperimental conditions:")
print(f"Non-gamified certain  : {n_no_game}")
print(f"Gamified certain      : {n_game_certain}")
print(f"Gamified chance-based : {n_game_cb}")
print(f"Missing condition     : {n_condition_missing}")

print("\nAttention check:")
print(f"Passed                : {n_attention_pass}")
print(f"Failed                : {n_attention_fail}")
print(f"Missing               : {n_attention_missing}")

print("\nHoneypot:")
print(f"Detected entries      : {n_honeypot}")


## 1.4 Apply Exclusion Criteria

The predefined exclusion criteria are applied sequentially: incomplete questionnaires, missing experimental condition, failed attention check, and honeypot indications. Each criterion is applied only after the preceding exclusions.


In [ ]:
# Apply Exclusion Criteria

# Start with all participant responses
df_clean = df_participants.copy()

# Initial sample size
n_initial = len(df_clean)

# 1. Remove incomplete responses

df_clean = df_clean[df_clean["Finished"] == 1].copy()

n_after_completion = len(df_clean)
n_removed_completion = n_initial - n_after_completion

# 2. Remove responses with missing experimental condition

df_clean = df_clean[df_clean["Condition"].notna()].copy()

n_after_condition = len(df_clean)
n_removed_condition = n_after_completion - n_after_condition

# ------------------------------------------------------------
# 3. Remove failed attention checks
# Correct response: 6 = "Agree"
# ------------------------------------------------------------

df_clean = df_clean[df_clean["Control_annoyance _4"] == 6].copy()

n_after_attention = len(df_clean)
n_removed_attention = n_after_condition - n_after_attention

# 4. Inspect honeypot field

n_honeypot = df_clean["Honeypot"].notna().sum()

# Exclusion summary

print("EXCLUSION SUMMARY")
print("-" * 50)

print(f"Initial participant responses      : {n_initial}")

print(f"\nRemoved (incomplete)               : {n_removed_completion}")
print(f"Remaining responses                : {n_after_completion}")

print(f"\nRemoved (missing condition)        : {n_removed_condition}")
print(f"Remaining responses                : {n_after_condition}")

print(f"\nRemoved (failed attention check)   : {n_removed_attention}")
print(f"Remaining responses                : {n_after_attention}")

print(f"\nHoneypot detections                : {n_honeypot}")

print("\nFinal analytical sample")
print(f"N = {len(df_clean)} participants")


## 1.5 Create the Final Analysis Dataset

After applying the predefined exclusions, the remaining participant responses are stored as the final analytical dataset used in all subsequent analyses.


In [ ]:
# Create and Save Final Analysis Dataset

# Create final analytical dataset
df = df_clean.copy()

# Reset row index
df.reset_index(drop=True, inplace=True)

# Define output file
ANALYSIS_DATA_FILE = OUTPUT_DIR / "Final_Analysis_Data.csv"

# Save cleaned dataset
df.to_csv(ANALYSIS_DATA_FILE, index=False)

print("✓ Final analysis dataset created and saved successfully.\n")

print(f"Observations : {len(df)}")
print(f"Variables    : {df.shape[1]}")

print(f"\nSaved as: {ANALYSIS_DATA_FILE.name}")


# 2. Variable Construction

This chapter prepares the study variables by standardizing names, defining the constructs, converting responses to numeric format, reverse-coding the required items, and computing the composite scores used in the analyses.


## 2.1 Standardize Variable Names

Long or automatically generated Qualtrics variable names are shortened for readability and consistency. Only the analysis labels change; the underlying data remain unchanged.


In [ ]:
# Standardize Variable Names

rename_dict = {

    # Manipulation checks
    "Manipulation_gamef.\xa0_1": "Gamefulness_1",
    "Manipulation_gamef.\xa0_2": "Gamefulness_2",

    "Manipulation_uncer.\xa0_1": "Uncertainty_1",
    "Manipulation_uncer.\xa0_2": "Uncertainty_2",

    "Manipulation_value\xa0_1": "Discount_Value_1",
    "Manipulation_value\xa0_2": "Discount_Value_2",

    # Product controls
    "C_product_controls\xa0_1": "Product_Involvement_1",
    "C_product_controls\xa0_2": "Product_Involvement_2",
    "C_product_controls\xa0_3": "Current_Product_Need",
    "C_product_controls\xa0_4": "Product_Familiarity",

    # Other controls
    "C_shopping_freq": "Shopping_Frequency",
    "Gaming_experience_1": "Gaming_Experience",

    # Demographics
    "Gender ": "Gender",
    "Consent ": "Consent"
}

df.rename(columns=rename_dict, inplace=True)

print("✓ Variable names standardized successfully.")


## 2.2 Define Analysis Constructs

Questionnaire items are grouped by theoretical construct. These definitions are reused for numeric conversion, reverse coding, composite scores, reliability analyses, descriptives, and hypothesis tests to ensure consistency.


In [ ]:
# Define Analysis Constructs

constructs = {

    # --------------------------------------------------------
    # Mediator
    # --------------------------------------------------------

    "Excitement": [
        "Excitement _1",
        "Excitement _2",
        "Excitement _3",
        "Excitement _4"
    ],

    # --------------------------------------------------------
    # Dependent Variables
    # --------------------------------------------------------

    "Urge_to_Buy": [
        "Urge_to_buy _1",
        "Urge_to_buy _2",
        "Urge_to_buy _3",
        "Urge_to_buy _4"
    ],

    "Purchase_Intention": [
        "Purchase_Intention_1",
        "Purchase_Intention_2",
        "Purchase_Intention_3"
    ],

    # --------------------------------------------------------
    # Moderator
    # --------------------------------------------------------

    "IBT_Cognitive": [
        "IBT_cognitive _1",
        "IBT_cognitive _2",
        "IBT_cognitive _3",
        "IBT_cognitive _4"
    ],

    "IBT_Affective": [
        "IBT_affective_1",
        "IBT_affective_2",
        "IBT_affective_3",
        "IBT_affective_4"
    ],

    # --------------------------------------------------------
    # Manipulation Checks
    # --------------------------------------------------------

    "Gamefulness": [
        "Gamefulness_1",
        "Gamefulness_2"
    ],

    "Reward_Uncertainty": [
        "Uncertainty_1",
        "Uncertainty_2"
    ],

    "Discount_Value": [
        "Discount_Value_1",
        "Discount_Value_2"
    ],

    # --------------------------------------------------------
    # Robustness Variable
    # --------------------------------------------------------

    "Annoyance": [
        "Control_annoyance _1",
        "Control_annoyance _2",
        "Control_annoyance _3"
    ],

    # --------------------------------------------------------
    # Control Variables
    # --------------------------------------------------------

    "Product_Involvement": [
        "Product_Involvement_1",
        "Product_Involvement_2"
    ]

}

# Additional Variables

single_item_controls = [
    "Current_Product_Need",
    "Product_Familiarity",
    "Shopping_Frequency",
    "Gaming_Experience"
]

demographic_variables = [
    "Age",
    "Gender"
]

condition_variable = "Condition"

attention_check = "Control_annoyance _4"

honeypot_variable = "Honeypot"

# Derived Variable Lists

all_scale_items = []

for items in constructs.values():
    all_scale_items.extend(items)

ibt_items = (
    constructs["IBT_Cognitive"] +
    constructs["IBT_Affective"]
)

print("✓ Analysis constructs defined successfully.\n")

print(f"Constructs defined          : {len(constructs)}")
print(f"Scale items                : {len(all_scale_items)}")
print(f"Single-item controls       : {len(single_item_controls)}")
print(f"Demographic variables      : {len(demographic_variables)}")


## 2.3 Convert Variables to Numeric

Multi-item constructs, single-item control variables, and age are converted to numeric format. Categorical variables such as gender and experimental condition remain unchanged.


In [ ]:
# Convert Variables to Numeric

# Collect all variables that should be numeric
numeric_variables = all_scale_items + single_item_controls + ["Age"]

# Remove duplicates while preserving order
numeric_variables = list(dict.fromkeys(numeric_variables))

# Convert variables
for column in numeric_variables:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Verification

numeric_summary = (
    df[numeric_variables]
    .dtypes
    .value_counts()
    .rename_axis("Data Type")
    .to_frame("Variables")
)

print("✓ Variables converted to numeric successfully.\n")

print(f"Variables converted : {len(numeric_variables)}\n")

display(numeric_summary)


## 2.4 Reverse-Code Items

Three cognitive Impulse Buying Tendency items are reverse-coded so that higher values consistently represent stronger impulse buying tendency. Because the items use seven-point scales, the transformation is `8 − x`.


In [ ]:
# Reverse-Code Items

reverse_items = [
    "IBT_cognitive _2",
    "IBT_cognitive _3",
    "IBT_cognitive _4"
]

# Apply reverse coding for 7-point Likert scales
for column in reverse_items:
    df[column] = 8 - df[column]

# Verify resulting value ranges
reverse_check = pd.DataFrame({
    "Minimum": df[reverse_items].min(),
    "Maximum": df[reverse_items].max()
})

print("✓ Reverse coding completed successfully.\n")
print(f"Reverse-coded items: {len(reverse_items)}")

display(reverse_check)


## 2.5 Compute Composite Scores

Composite scores are calculated as the arithmetic mean of their respective items, preserving the original seven-point metric. Cognitive and affective IBT scores are retained separately, and overall IBT is calculated as the mean of all eight IBT items.


In [ ]:
# Compute Composite Scores

# Compute mean scores for all multi-item constructs
for construct, items in constructs.items():
    df[construct] = df[items].mean(axis=1)

# Compute overall Impulse Buying Tendency (IBT)
df["IBT"] = df[ibt_items].mean(axis=1)

# Verification

composite_variables = list(constructs.keys()) + ["IBT"]

summary = pd.DataFrame({
    "Minimum": df[composite_variables].min(),
    "Maximum": df[composite_variables].max(),
    "Mean": df[composite_variables].mean().round(2)
})

print("✓ Composite scores computed successfully.\n")

print(f"Composite variables created: {len(composite_variables)}\n")

display(summary)


## 2.6 Chapter Summary

Variable names are standardized, relevant responses are converted to numeric format, the required IBT items are reverse-coded, and composite scores are calculated for all multi-item constructs. The resulting variables are used in the scale evaluation and hypothesis tests.


# 3. Scale Evaluation and Preliminary Analyses

This chapter evaluates the prepared dataset before hypothesis testing. It examines internal consistency, descriptive statistics, correlations, manipulation checks, sample characteristics, and randomization checks.


## 3.1 Overview of the Analysis Dataset

The final dataset is inspected for observations, variables, missing values, and duplicate records before evaluating the measures.


In [ ]:
# Overview of the Analysis Dataset

# Variables used in the statistical analyses
analysis_variables = (
    composite_variables +
    single_item_controls +
    ["IBT", "Age", "Gender", "Condition"]
)

# Remove duplicates while preserving order
analysis_variables = list(dict.fromkeys(analysis_variables))

# Create overview table
overview = pd.DataFrame({
    "Metric": [
        "Observations",
        "Analysis variables",
        "Missing values (analysis variables)",
        "Duplicate rows"
    ],
    "Value": [
        len(df),
        len(analysis_variables),
        df[analysis_variables].isna().sum().sum(),
        df.duplicated().sum()
    ]
})

print("✓ Final analysis dataset verified.\n")

display(overview)


## 3.2 Internal Consistency Assessment

Cronbach's alpha is calculated for constructs with three or more items. For two-item measures, the inter-item Pearson correlation is reported because coefficient alpha is less informative with only two indicators.


In [ ]:
# Internal Consistency Assessment

reliability_results = []

for construct, items in constructs.items():

    n_items = len(items)

    # --------------------------------------------------------
    # Three or more items: Cronbach's alpha
    # --------------------------------------------------------

    if n_items >= 3:

        alpha, _ = pg.cronbach_alpha(
            data=df[items]
        )

        reliability_results.append({
            "Construct": construct,
            "Items": n_items,
            "Statistic": "α",
            "Coefficient": alpha
        })

    # --------------------------------------------------------
    # Two-item measures: Pearson inter-item correlation
    # --------------------------------------------------------

    elif n_items == 2:

        r, _ = stats.pearsonr(
            df[items[0]],
            df[items[1]]
        )

        reliability_results.append({
            "Construct": construct,
            "Items": n_items,
            "Statistic": "r",
            "Coefficient": r
        })

# Overall IBT

ibt_alpha, _ = pg.cronbach_alpha(
    data=df[ibt_items]
)

reliability_results.append({
    "Construct": "IBT",
    "Items": len(ibt_items),
    "Statistic": "α",
    "Coefficient": ibt_alpha
})

# Create final reliability table

reliability_table = pd.DataFrame(reliability_results)

reliability_table["Coefficient"] = (
    reliability_table["Coefficient"]
    .round(2)
)

print("✓ Internal consistency assessed successfully.\n")

display(reliability_table)


## 3.3 Descriptive Statistics

Means, standard deviations, minima, and maxima are calculated for all composite constructs used in the subsequent analyses.


In [ ]:
# Descriptive Statistics

# Composite variables included in the analyses
descriptive_variables = [
    "Excitement",
    "Urge_to_Buy",
    "Purchase_Intention",
    "IBT_Cognitive",
    "IBT_Affective",
    "IBT",
    "Gamefulness",
    "Reward_Uncertainty",
    "Discount_Value",
    "Annoyance",
    "Product_Involvement"
]

# Display names used in output tables
display_names = {
    "Excitement": "Excitement",
    "Urge_to_Buy": "Urge to Buy",
    "Purchase_Intention": "Purchase Intention",
    "IBT_Cognitive": "IBT (Cognitive)",
    "IBT_Affective": "IBT (Affective)",
    "IBT": "Impulse Buying Tendency",
    "Gamefulness": "Perceived Gamefulness",
    "Reward_Uncertainty": "Perceived Reward Uncertainty",
    "Discount_Value": "Perceived Discount Value",
    "Annoyance": "Annoyance / Friction",
    "Product_Involvement": "Product Involvement"
}

# Calculate descriptive statistics
descriptive_table = (
    df[descriptive_variables]
    .agg(["mean", "std", "min", "max"])
    .T
    .rename(columns={
        "mean": "Mean",
        "std": "SD",
        "min": "Minimum",
        "max": "Maximum"
    })
)

# Apply readable construct names
descriptive_table.rename(index=display_names, inplace=True)

print("✓ Descriptive statistics calculated successfully.\n")

display(
    descriptive_table.style.format({
        "Mean": "{:.2f}",
        "SD": "{:.2f}",
        "Minimum": "{:.2f}",
        "Maximum": "{:.2f}"
    })
)


## 3.4 Correlation Matrix

Pearson correlations are calculated for the principal study constructs. The table reports each construct's mean and standard deviation together with the lower-triangular correlation matrix; significance is indicated by *p* < .05, **p** < .01, and ***p*** < .001.


In [ ]:
# Correlation Matrix

corr_variables = descriptive_variables

# Reuse display names for presentation

# Numbered row labels
numbered_labels = {
    variable: f"{i}. {display_names[variable]}"
    for i, variable in enumerate(corr_variables, start=1)
}

# Empty lower-triangular matrix
corr_table = pd.DataFrame(
    "",
    index=[numbered_labels[v] for v in corr_variables],
    columns=[str(i) for i in range(1, len(corr_variables) + 1)]
)

# Calculate Pearson correlations
for i, var1 in enumerate(corr_variables):

    for j, var2 in enumerate(corr_variables):

        if i == j:
            corr_table.iloc[i, j] = "—"

        elif i > j:

            r, p = stats.pearsonr(df[var1], df[var2])

            if p < .001:
                stars = "***"
            elif p < .01:
                stars = "**"
            elif p < .05:
                stars = "*"
            else:
                stars = ""

            # Format correlation and remove leading zero
            r_string = f"{r:.2f}"

            if r_string.startswith("0"):
                r_string = r_string[1:]

            elif r_string.startswith("-0"):
                r_string = "-" + r_string[2:]

            corr_table.iloc[i, j] = f"{r_string}{stars}"

# Means and standard deviations
means = df[corr_variables].mean()
sds = df[corr_variables].std()

correlation_table = corr_table.copy()

correlation_table.insert(
    0,
    "SD",
    [f"{sds[v]:.2f}" for v in corr_variables]
)

correlation_table.insert(
    0,
    "M",
    [f"{means[v]:.2f}" for v in corr_variables]
)

print("✓ Correlation matrix calculated successfully.\n")

display(correlation_table)

print("\nNote. Pearson correlations are reported.")
print("* p < .05, ** p < .01, *** p < .001.")


## 3.5 Manipulation Checks

Perceived gamefulness, perceived reward uncertainty, and perceived discount value are compared across the three experimental conditions. Condition-level descriptives are reported first, followed by between-group tests and post hoc comparisons where appropriate.


In [ ]:
# Setup for Manipulation Checks

condition_mapping = {
    "No_Game": "Non-Gamified Certain",
    "Game_Certain": "Gamified Certain",
    "Game_CB": "Gamified Chance-Based"
}

df["Condition_Label"] = (
    df["Condition"]
    .map(condition_mapping)
)

# Display order

condition_order = [
    "Non-Gamified Certain",
    "Gamified Certain",
    "Gamified Chance-Based"
]

# Manipulation-check variables

manipulation_variables = [
    "Gamefulness",
    "Reward_Uncertainty",
    "Discount_Value"
]

print("✓ Manipulation-check setup completed successfully.")


### 3.5.1 Descriptive Statistics by Experimental Condition

Means, standard deviations, and sample sizes are reported for each manipulation-check variable by experimental condition.


In [ ]:
# Create presentation table

presentation_table = pd.DataFrame(index=condition_order)

presentation_table["n"] = (
    df.groupby("Condition_Label", observed=True)
      .size()
      .reindex(condition_order)
)

for variable in manipulation_variables:

    label = display_names[variable]

    means = (
        df.groupby("Condition_Label", observed=True)[variable]
          .mean()
          .reindex(condition_order)
    )

    sds = (
        df.groupby("Condition_Label", observed=True)[variable]
          .std()
          .reindex(condition_order)
    )

    presentation_table[label] = [
        f"{m:.2f} ({sd:.2f})"
        for m, sd in zip(means, sds)
    ]

display(presentation_table)


### 3.5.2 Between-Group Comparisons

One-way ANOVAs test whether the three experimental conditions differ on the manipulation-check variables. Significant omnibus tests are followed by post hoc comparisons.


In [ ]:
# One-Way ANOVAs for Manipulation Checks

anova_results = []

for variable in manipulation_variables:

    groups = [
        df.loc[df["Condition"] == condition, variable]
        for condition in [
            "No_Game",
            "Game_Certain",
            "Game_CB"
        ]
    ]

    F, p = stats.f_oneway(*groups)

    anova_results.append({
        "Variable": display_names[variable],
        "F": round(F, 2),
        "p": p
    })

anova_table = pd.DataFrame(anova_results)

# APA formatting for p-values
anova_table["p"] = anova_table["p"].apply(
    lambda x: "< .001" if x < .001 else f"{x:.3f}"
)

print("✓ One-way ANOVAs completed successfully.\n")

display(anova_table)


### 3.5.3 Post Hoc Comparisons and Effect Sizes

For manipulation-check variables with significant omnibus effects, Tukey's Honestly Significant Difference (HSD) test identifies pairwise differences. Eta squared (η²) quantifies the variance explained by experimental condition. No post hoc comparisons are conducted for non-significant omnibus effects.


In [ ]:
# Tukey HSD Post Hoc Comparisons


# Variables with significant omnibus ANOVA results
significant_variables = [
    "Gamefulness",
    "Reward_Uncertainty"
]

# Readable condition labels
condition_labels = {
    "No_Game": "Non-gamified certain",
    "Game_Certain": "Gamified certain",
    "Game_CB": "Gamified chance-based"
}

# Logical comparison order
comparison_pairs = [
    ("No_Game", "Game_Certain"),
    ("No_Game", "Game_CB"),
    ("Game_Certain", "Game_CB")
]

posthoc_results = []

for variable in significant_variables:

    tukey = pairwise_tukeyhsd(
        endog=df[variable],
        groups=df["Condition"],
        alpha=0.05
    )

    tukey_df = pd.DataFrame(
        tukey._results_table.data[1:],
        columns=tukey._results_table.data[0]
    )

    for group1, group2 in comparison_pairs:

        # Locate the relevant Tukey comparison
        row = tukey_df[
            (
                (tukey_df["group1"] == group1) &
                (tukey_df["group2"] == group2)
            )
            |
            (
                (tukey_df["group1"] == group2) &
                (tukey_df["group2"] == group1)
            )
        ].iloc[0]

        # Express difference consistently as Group 2 - Group 1
        mean_difference = (
            df.loc[df["Condition"] == group2, variable].mean()
            -
            df.loc[df["Condition"] == group1, variable].mean()
        )

        p_value = float(row["p-adj"])

        posthoc_results.append({
            "Variable": display_names[variable],
            "Group 1": condition_labels[group1],
            "Group 2": condition_labels[group2],
            "Mean Difference": mean_difference,
            "p": p_value,
            "Significant": bool(row["reject"])
        })

posthoc_table = pd.DataFrame(posthoc_results)

# Format output
posthoc_table["Mean Difference"] = (
    posthoc_table["Mean Difference"]
    .round(2)
)

posthoc_table["p"] = posthoc_table["p"].apply(
    lambda x: "< .001" if x < .001 else f"{x:.3f}"
)

posthoc_table["Significant"] = (
    posthoc_table["Significant"]
    .map({
        True: "Yes",
        False: "No"
    })
)

print("✓ Tukey post hoc comparisons completed successfully.\n")

display(posthoc_table)


In [ ]:
# Eta-Squared Effect Sizes

effect_size_results = []

for variable in manipulation_variables:

    grand_mean = df[variable].mean()

    # Sum of squares between conditions
    ss_between = sum(
        len(group) * (group.mean() - grand_mean) ** 2
        for _, group in df.groupby("Condition")[variable]
    )

    # Total sum of squares
    ss_total = ((df[variable] - grand_mean) ** 2).sum()

    eta_squared = ss_between / ss_total

    # Conventional interpretation
    if eta_squared >= 0.14:
        magnitude = "Large"
    elif eta_squared >= 0.06:
        magnitude = "Medium"
    elif eta_squared >= 0.01:
        magnitude = "Small"
    else:
        magnitude = "Negligible"

    effect_size_results.append({
        "Variable": display_names[variable],
        "η²": eta_squared,
        "Magnitude": magnitude
    })

effect_size_table = pd.DataFrame(effect_size_results)

effect_size_table["η²"] = (
    effect_size_table["η²"]
    .round(3)
)

print("✓ Effect sizes calculated successfully.\n")

display(effect_size_table)


## 3.6 Sample Characteristics

Age, gender, and the distribution of participants across the experimental conditions are summarized for the final analytical sample.


In [ ]:
# Sample Characteristics

# Age

age_summary = pd.DataFrame({
    "Statistic": [
        "Mean age",
        "Standard deviation",
        "Minimum",
        "Maximum"
    ],
    "Value": [
        df["Age"].mean(),
        df["Age"].std(),
        df["Age"].min(),
        df["Age"].max()
    ]
})

# Gender

gender_labels = {
    1: "Male",
    2: "Female",
    4: "Other / undisclosed",
    "1": "Male",
    "2": "Female",
    "4": "Other / undisclosed"
}

# Add readable gender label to dataset
df["Gender_Label"] = df["Gender"].replace(gender_labels)

gender_table = (
    df["Gender_Label"]
    .value_counts()
    .rename_axis("Gender")
    .reset_index(name="n")
)

gender_table["%"] = (
    gender_table["n"] / len(df) * 100
)

# Preserve logical order
gender_order = [
    "Male",
    "Female",
    "Other / undisclosed"
]

gender_table["Gender"] = pd.Categorical(
    gender_table["Gender"],
    categories=gender_order,
    ordered=True
)

gender_table = (
    gender_table
    .sort_values("Gender")
    .reset_index(drop=True)
)

# Experimental Conditions

condition_table = (
    df["Condition_Label"]
    .value_counts()
    .rename_axis("Condition")
    .reset_index(name="n")
)

condition_table["%"] = (
    condition_table["n"] / len(df) * 100
)


condition_table["Condition"] = pd.Categorical(
    condition_table["Condition"],
    categories=condition_order,
    ordered=True
)

condition_table = (
    condition_table
    .sort_values("Condition")
    .reset_index(drop=True)
)

# Display

print("✓ Sample characteristics calculated successfully.\n")

print("Age")
display(
    age_summary.style.format({
        "Value": "{:.2f}"
    })
)

print("Gender")
display(
    gender_table.style.format({
        "%": "{:.1f}"
    })
)

print("Experimental conditions")
display(
    condition_table.style.format({
        "%": "{:.1f}"
    })
)


## 3.7 Randomization Checks

Age and gender are compared across experimental conditions to assess whether random assignment produced demographically comparable groups.


### 3.7.1 Age

A one-way ANOVA compares age across the three experimental conditions. A non-significant result indicates no systematic age differences between groups before the manipulation.


In [ ]:
# Randomization Check: Age


# One-way ANOVA

age_model = smf.ols(
    "Age ~ C(Condition_Label)",
    data=df
).fit()

age_anova = anova_lm(age_model, typ=2)

# Create APA-style results table

age_randomization_table = pd.DataFrame({

    "Statistic": ["F", "df1", "df2", "p-value"],

    "Value": [

        round(age_anova.loc["C(Condition_Label)", "F"], 2),

        int(age_anova.loc["C(Condition_Label)", "df"]),

        int(age_anova.loc["Residual", "df"]),

        "< .001"
        if age_anova.loc["C(Condition_Label)", "PR(>F)"] < .001
        else f"{age_anova.loc['C(Condition_Label)', 'PR(>F)']:.3f}"

    ]

})

# Display

print("✓ Age randomization check completed.\n")

display(age_randomization_table)


### 3.7.2 Gender

A chi-square test of independence compares gender across conditions. Because only one participant reported another gender or preferred not to disclose gender, the test is restricted to male and female participants to satisfy the chi-square assumptions.


In [ ]:
# Randomization Check: Gender


# Restrict analysis to male and female participants

gender_randomization = df[
    df["Gender_Label"].isin([
        "Male",
        "Female"
    ])
].copy()

# Contingency table

gender_crosstab = pd.crosstab(
    gender_randomization["Gender_Label"],
    gender_randomization["Condition_Label"]
)

# Chi-square test

chi2, p, dof, expected = chi2_contingency(gender_crosstab)

# APA-style results table

gender_randomization_table = pd.DataFrame({

    "Statistic": [
        "Chi-square",
        "df",
        "p-value"
    ],

    "Value": [

        round(chi2, 2),

        dof,

        "< .001"
        if p < .001
        else f"{p:.3f}"

    ]

})

# Display

print("✓ Gender randomization check completed.\n")

print("Observed frequencies")
display(gender_crosstab)

print("Expected frequencies")
display(
    pd.DataFrame(
        expected,
        index=gender_crosstab.index,
        columns=gender_crosstab.columns
    ).round(2)
)

print("Test results")
display(gender_randomization_table)


## 3.8 Save Analysis-Ready Dataset

The variables required for the empirical analyses are saved as a separate analysis-ready dataset. Administrative Qualtrics fields not needed for the analyses are excluded.


In [ ]:
# Save Analysis-Ready Dataset

analysis_ready_variables = (
    all_scale_items +
    single_item_controls +
    [
        "Age",
        "Gender",
        "Gender_Label",
        "Condition",
        "Condition_Label"
    ] +
    composite_variables
)

# Remove duplicates while preserving order
analysis_ready_variables = list(
    dict.fromkeys(analysis_ready_variables)
)

# Create reduced analysis-ready dataset
df_analysis = df[analysis_ready_variables].copy()

# Define output file
ANALYSIS_READY_FILE = OUTPUT_DIR / "Analysis_Ready_Data.csv"

# Save dataset
df_analysis.to_csv(
    ANALYSIS_READY_FILE,
    index=False
)

print("✓ Analysis-ready dataset saved successfully.\n")

print(f"Observations : {len(df_analysis)}")
print(f"Variables    : {df_analysis.shape[1]}")
print(f"Missing values: {df_analysis.isna().sum().sum()}")

print(f"\nSaved as: {ANALYSIS_READY_FILE.name}")


# 4. Empirical Results

This chapter tests the hypotheses. H1 examines condition differences in excitement, H2 tests the associations of excitement with urge to buy and purchase intention, H3 tests the indirect effects of promotion condition on both consumer outcomes through excitement, and H4 tests whether these indirect effects depend on impulse buying tendency (IBT).


## 4.1 Overview of Hypothesis Testing

Unless stated otherwise, all tests are two-sided with α = .05. H1a and H1b are tested with a one-way ANOVA followed by planned independent-samples *t*-tests, Cohen's *d*, and 95% confidence intervals for the raw mean differences. H2a and H2b are tested with simple linear regressions. H3a and H3b test mediation by estimating the indirect effects of the two planned promotion contrasts on urge to buy and purchase intention through excitement, using 10,000 nonparametric bootstrap resamples and percentile confidence intervals. H4a and H4b use a second-stage moderated mediation approach corresponding to Hayes' (2022) PROCESS Model 14 logic, with promotion condition, excitement, IBT, and the excitement × IBT interaction in the outcome models. Conditional indirect effects are bootstrapped, and moderated mediation is evaluated with the index of moderated mediation.


In [ ]:
# Analysis Setup

# Experimental conditions (reference order)
analysis_condition_order = [
    "No_Game",
    "Game_Certain",
    "Game_CB"
]

condition_labels = {
    "No_Game": "Non-gamified certain",
    "Game_Certain": "Gamified certain",
    "Game_CB": "Gamified chance-based"
}

# Outcome variables
outcomes = [
    "Urge_to_Buy",
    "Purchase_Intention"
]

print("✓ Analysis setup completed successfully.")


### 4.1.1 Assumption Checks

Before hypothesis testing, ANOVA assumptions are assessed using homogeneity of variance and residual normality. Regression diagnostics assess normality, homoscedasticity, independence, linearity, and multicollinearity. Robust alternatives are additionally estimated where appropriate.


#### ANOVA Assumptions


In [ ]:
# Assumption Checks: Analysis of Variance


anova_vars = [
    "Gamefulness",
    "Reward_Uncertainty",
    "Discount_Value",
    "Excitement"
]

anova_assumption_rows = []

for v in anova_vars:

    groups = [
        df.loc[df["Condition"] == c, v].dropna()
        for c in ["No_Game", "Game_Certain", "Game_CB"]
    ]

    # Homogeneity of variance
    lev_W, lev_p = levene(*groups, center="median")

    # Normality of within-group residuals
    residuals = pd.concat(
        [g - g.mean() for g in groups],
        ignore_index=True
    )

    sw_W, sw_p = shapiro(residuals)

    anova_assumption_rows.append({
        "Variable": v,
        "Levene W": round(lev_W, 2),
        "Levene p": "< .001" if lev_p < .001 else f"{lev_p:.3f}",
        "Shapiro-Wilk W": round(sw_W, 3),
        "Shapiro-Wilk p": "< .001" if sw_p < .001 else f"{sw_p:.3f}"
    })

anova_assumptions = pd.DataFrame(anova_assumption_rows)

print("✓ ANOVA assumption checks completed.\n")
display(anova_assumptions)


#### Regression Assumptions


In [ ]:
# Assumption Checks: Regression Models


# Centered predictors and interaction term
df["Excitement_c"] = df["Excitement"] - df["Excitement"].mean()
df["IBT_c"] = df["IBT"] - df["IBT"].mean()
df["Excitement_x_IBT"] = df["Excitement_c"] * df["IBT_c"]

# Condition indicators
df["D_Game_Certain"] = (df["Condition"] == "Game_Certain").astype(int)
df["D_Game_CB"] = (df["Condition"] == "Game_CB").astype(int)

regression_models = {
    "H2a: Urge to Buy":
        smf.ols("Urge_to_Buy ~ Excitement", data=df).fit(),

    "H2b: Purchase Intention":
        smf.ols("Purchase_Intention ~ Excitement", data=df).fit(),

    "H4a: Urge to Buy":
        smf.ols(
            "Urge_to_Buy ~ D_Game_Certain + D_Game_CB + "
            "Excitement_c + IBT_c + Excitement_x_IBT",
            data=df
        ).fit(),

    "H4b: Purchase Intention":
        smf.ols(
            "Purchase_Intention ~ D_Game_Certain + D_Game_CB + "
            "Excitement_c + IBT_c + Excitement_x_IBT",
            data=df
        ).fit()
}

regression_assumption_rows = []

for name, model in regression_models.items():

    resid = model.resid

    # Normality
    sw_W, sw_p = shapiro(resid)

    # Homoscedasticity
    _, bp_p, _, _ = het_breuschpagan(
        resid,
        model.model.exog
    )

    # Independence
    dw = durbin_watson(resid)

    # Multicollinearity
    exog = model.model.exog
    names = model.model.exog_names

    vifs = [
        variance_inflation_factor(exog, i)
        for i, n in enumerate(names)
        if n != "Intercept"
    ]

    max_vif = max(vifs) if vifs else float("nan")

    regression_assumption_rows.append({
        "Model": name,
        "Shapiro-Wilk W": round(sw_W, 3),
        "Shapiro-Wilk p": "< .001" if sw_p < .001 else f"{sw_p:.3f}",
        "Breusch-Pagan p": "< .001" if bp_p < .001 else f"{bp_p:.3f}",
        "Durbin-Watson": round(dw, 2),
        "Max VIF": round(max_vif, 2)
    })

regression_assumptions = pd.DataFrame(regression_assumption_rows)

print("✓ Regression assumption checks completed.\n")
display(regression_assumptions)


#### Regression Diagnostic Plots


In [ ]:
# Regression Diagnostic Plots


for name, model in regression_models.items():

    # Residuals vs fitted values
    plt.figure(figsize=(6, 4))
    plt.scatter(model.fittedvalues, model.resid, alpha=0.6)
    plt.axhline(0, linestyle="--")
    plt.title(f"Residuals vs. Fitted: {name}")
    plt.xlabel("Fitted Values")
    plt.ylabel("Residuals")
    plt.show()

    # Q-Q plot
    sm.qqplot(model.resid, line="45", fit=True)
    plt.title(f"Q-Q Plot: {name}")
    plt.show()


#### Assumption-Robust Confirmation


In [ ]:
# Assumption-Robust Confirmation


# Welch ANOVA
welch_rows = []

for v in anova_vars:

    w = pg.welch_anova(
        data=df,
        dv=v,
        between="Condition"
    )

    welch_rows.append({
        "Variable": v,
        "Welch F": round(w.loc[0, "F"], 2),
        "df1": int(w.loc[0, "ddof1"]),
        "df2": round(w.loc[0, "ddof2"], 1),
        "p": "< .001" if w.loc[0, "p_unc"] < .001
             else f"{w.loc[0, 'p_unc']:.3f}"
    })

welch_anova_table = pd.DataFrame(welch_rows)

# Games-Howell post hoc for excitement
gameshowell_excitement = pg.pairwise_gameshowell(
    data=df,
    dv="Excitement",
    between="Condition"
)[["A", "B", "diff", "pval"]]

# HC3 robust regressions
hc3_rows = []

for label, formula, focal_predictor in [
    ("H2a", "Urge_to_Buy ~ Excitement", "Excitement"),
    ("H2b", "Purchase_Intention ~ Excitement", "Excitement"),
    (
        "H4a",
        "Urge_to_Buy ~ D_Game_Certain + D_Game_CB + "
        "Excitement_c + IBT_c + Excitement_x_IBT",
        "Excitement_x_IBT"
    ),
    (
        "H4b",
        "Purchase_Intention ~ D_Game_Certain + D_Game_CB + "
        "Excitement_c + IBT_c + Excitement_x_IBT",
        "Excitement_x_IBT"
    )
]:

    m = smf.ols(formula, data=df).fit(cov_type="HC3")

    p = m.pvalues[focal_predictor]

    hc3_rows.append({
        "Hypothesis": label,
        "B": round(m.params[focal_predictor], 3),
        "Robust SE": round(m.bse[focal_predictor], 3),
        "t": round(m.tvalues[focal_predictor], 2),
        "p": "< .001" if p < .001 else f"{p:.3f}"
    })

hc3_table = pd.DataFrame(hc3_rows)

print("✓ Assumption-robust confirmation completed.\n")

print("Welch ANOVA")
display(welch_anova_table)

print("\nGames-Howell post hoc (Excitement)")
display(gameshowell_excitement.round(3))

print("\nHC3 robust regression estimates")
display(hc3_table)


## 4.2 Effect of Promotion Condition on Excitement (H1)

An omnibus one-way ANOVA first tests whether excitement differs across the three promotion conditions. Planned comparisons then test:

- **H1a:** Gamified chance-based promotions generate higher excitement than gamified certain-reward promotions.
- **H1b:** Gamified certain-reward promotions generate higher excitement than non-gamified certain promotions.


### 4.2.1 Overall Effect of Promotion Condition on Excitement

A one-way ANOVA tests whether at least one experimental condition differs in excitement. The planned comparisons are then used to test H1a and H1b directly.


In [ ]:
# H1: Overall Effect of Promotion Condition on Excitement

groups = [
    df.loc[df["Condition"] == condition, "Excitement"]
    for condition in analysis_condition_order
]

F, p = stats.f_oneway(*groups)

# Effect size: eta-squared (between-group SS / total SS)
grand_mean = df["Excitement"].mean()
ss_between = sum(
    len(g) * (g.mean() - grand_mean) ** 2
    for g in groups
)
ss_total = ((df["Excitement"] - grand_mean) ** 2).sum()
eta_squared = ss_between / ss_total

anova_summary = pd.DataFrame({
    "Statistic": [
        "F",
        "df",
        "p",
        "η²"
    ],
    "Value": [
        round(F, 2),
        f"({len(groups) - 1}, {len(df) - len(groups)})",
        "< .001" if p < .001 else f"{p:.3f}",
        round(eta_squared, 2)
    ]
})

print("✓ Omnibus ANOVA completed successfully.\n")

display(anova_summary)


### 4.2.2 Planned Pairwise Comparisons

Two-sided independent-samples *t*-tests evaluate the two a priori contrasts. Support requires statistical significance and an effect in the predicted direction:

- **H1a:** Gamified chance-based vs. gamified certain.
- **H1b:** Gamified certain vs. non-gamified certain.


In [ ]:
# H1: Planned Pairwise Comparisons


comparisons = [
    ("H1a",
     "Game_CB",
     "Game_Certain"),

    ("H1b",
     "Game_Certain",
     "No_Game")
]

results = []

for hypothesis, group1, group2 in comparisons:

    data1 = df.loc[df["Condition"] == group1, "Excitement"]
    data2 = df.loc[df["Condition"] == group2, "Excitement"]

    t, p = ttest_ind(
        data1,
        data2,
        equal_var=True
    )

    mean_difference = data1.mean() - data2.mean()

    results.append({

        "Hypothesis": hypothesis,

        "Comparison":
            f"{condition_labels[group1]} vs.\n{condition_labels[group2]}",

        "Mean Difference":
            round(mean_difference, 3),

        "t":
            round(t, 3),

        "p":
            "< .001" if p < .001 else f"{p:.3f}",

        "Supported":
            "Yes" if (p < .05 and mean_difference > 0) else "No"
    })

planned_comparisons = pd.DataFrame(results)

print("✓ Planned comparisons completed successfully.\n")

display(planned_comparisons)


### 4.2.3 Effect Sizes and Confidence Intervals

Cohen's *d* provides a standardized effect size for each planned comparison. The reported 95% confidence intervals refer to the corresponding raw mean differences (ΔM), not to Cohen's *d*.


In [ ]:
# H1: Standardized Effect Sizes and Confidence Intervals

comparisons = [
    ("H1a", "Game_CB", "Game_Certain"),
    ("H1b", "Game_Certain", "No_Game")
]

effect_results = []

for hypothesis, group1, group2 in comparisons:

    data1 = df.loc[df["Condition"] == group1, "Excitement"]
    data2 = df.loc[df["Condition"] == group2, "Excitement"]

    # Means
    mean1 = data1.mean()
    mean2 = data2.mean()

    # Sample sizes
    n1 = len(data1)
    n2 = len(data2)

    # Standard deviations
    sd1 = data1.std(ddof=1)
    sd2 = data2.std(ddof=1)

    # Pooled standard deviation
    pooled_sd = np.sqrt(
        ((n1 - 1) * sd1**2 + (n2 - 1) * sd2**2) /
        (n1 + n2 - 2)
    )

    # Cohen's d
    cohen_d = (mean1 - mean2) / pooled_sd

    # 95% confidence interval for the mean difference
    ci = stats.ttest_ind(
        data1,
        data2,
        equal_var=True
    ).confidence_interval()

    effect_results.append({
        "Hypothesis": hypothesis,
        "Comparison": (
            f"{condition_labels[group1]} vs. "
            f"{condition_labels[group2]}"
        ),
        "ΔM": round(mean1 - mean2, 3),
        "95% CI for ΔM": f"[{ci.low:.2f}, {ci.high:.2f}]",
        "t(df)": f"t({n1 + n2 - 2}) = {stats.ttest_ind(data1, data2, equal_var=True).statistic:.2f}",
        "p": ("< .001" if stats.ttest_ind(data1, data2, equal_var=True).pvalue < .001
              else f"{stats.ttest_ind(data1, data2, equal_var=True).pvalue:.3f}"),
        "Cohen's d": round(cohen_d, 3)
    })

effect_table = pd.DataFrame(effect_results)

print("✓ Standardized effect sizes and confidence intervals calculated successfully.\n")

display(effect_table)


## 4.3 Effect of Excitement on Consumer Responses (H2)

Simple linear regressions test whether excitement positively predicts urge to buy (H2a) and purchase intention (H2b).


### 4.3.1 Effect of Excitement on Urge to Buy (H2a)

A simple linear regression tests whether excitement positively predicts urge to buy. A significant positive coefficient supports H2a.


In [ ]:
# H2a: Excitement → Urge to Buy

model_h2a = smf.ols(
    "Urge_to_Buy ~ Excitement",
    data=df
).fit()

# Extract regression statistics
b = model_h2a.params["Excitement"]
se = model_h2a.bse["Excitement"]
t_value = model_h2a.tvalues["Excitement"]
p_value = model_h2a.pvalues["Excitement"]

# Standardized regression coefficient
beta = (
    b
    * df["Excitement"].std(ddof=1)
    / df["Urge_to_Buy"].std(ddof=1)
)

# 95% confidence interval
ci_low, ci_high = model_h2a.conf_int().loc["Excitement"]

# Create result table
h2a_table = pd.DataFrame({
    "Hypothesis": ["H2a"],
    "Predictor": ["Excitement"],
    "B": [b],
    "β": [beta],
    "SE": [se],
    "t": [t_value],
    "p": [
        "< .001" if p_value < .001 else f"{p_value:.3f}"
    ],
    "95% CI": [
        f"[{ci_low:.2f}, {ci_high:.2f}]"
    ],
    "R²": [model_h2a.rsquared],
    "Supported": [
        "Yes" if (p_value < .05 and b > 0) else "No"
    ]
})

# Round numerical values
h2a_table = h2a_table.round({
    "B": 3,
    "β": 3,
    "SE": 3,
    "t": 3,
    "R²": 3
})

print("✓ H2a regression completed successfully.\n")

display(h2a_table)


### 4.3.2 Effect of Excitement on Purchase Intention (H2b)

A simple linear regression tests whether excitement positively predicts purchase intention. A significant positive coefficient supports H2b.


In [ ]:
# H2b: Excitement → Purchase Intention

model_h2b = smf.ols(
    "Purchase_Intention ~ Excitement",
    data=df
).fit()

# Extract regression statistics
b = model_h2b.params["Excitement"]
se = model_h2b.bse["Excitement"]
t_value = model_h2b.tvalues["Excitement"]
p_value = model_h2b.pvalues["Excitement"]

# Standardized regression coefficient
beta = (
    b
    * df["Excitement"].std(ddof=1)
    / df["Purchase_Intention"].std(ddof=1)
)

# 95% confidence interval
ci_low, ci_high = model_h2b.conf_int().loc["Excitement"]

# Create result table
h2b_table = pd.DataFrame({
    "Hypothesis": ["H2b"],
    "Predictor": ["Excitement"],
    "B": [b],
    "β": [beta],
    "SE": [se],
    "t": [t_value],
    "p": [
        "< .001" if p_value < .001 else f"{p_value:.3f}"
    ],
    "95% CI": [
        f"[{ci_low:.2f}, {ci_high:.2f}]"
    ],
    "R²": [model_h2b.rsquared],
    "Supported": [
        "Yes" if (p_value < .05 and b > 0) else "No"
    ]
})

# Round numerical values
h2b_table = h2b_table.round({
    "B": 3,
    "β": 3,
    "SE": 3,
    "t": 3,
    "R²": 3
})

print("✓ H2b regression completed successfully.\n")

display(h2b_table)


## 4.4 Mediation Analysis (H3)

H3 tests positive contrast-specific indirect effects of promotion type on the two consumer outcomes through excitement. Because promotion condition has three levels, the same two planned contrasts used for H1 are evaluated separately and in the same order: (1) gamified chance-based vs. gamified certain and (2) gamified certain vs. non-gamified certain. For each outcome, the indirect effect is the product of the relevant condition-to-excitement path (*a*) and the excitement-to-outcome path controlling for promotion condition (*b*). Indirect effects are evaluated with 10,000 nonparametric bootstrap resamples and 95% percentile confidence intervals. Because H3 predicts positive indirect effects, support requires the entire bootstrap confidence interval to lie above zero.

**H3a:** Promotion type has a positive indirect effect on urge to buy through excitement, such that (i) gamified chance-based promotions compared with gamified certain-reward promotions and (ii) gamified certain-reward promotions compared with non-gamified certain promotions increase urge to buy through higher excitement.

**H3b:** Promotion type has a positive indirect effect on purchase intention through excitement, such that (i) gamified chance-based promotions compared with gamified certain-reward promotions and (ii) gamified certain-reward promotions compared with non-gamified certain promotions increase purchase intention through higher excitement.


### 4.4.1 Reusable Mediation Function

The function below estimates the mediator model and an outcome model containing promotion condition and excitement. It then bootstraps the two planned contrast-specific indirect effects. The same procedure is used for urge to buy (H3a) and purchase intention (H3b).


In [ ]:
def run_mediation(
    data,
    outcome,
    n_boot=10_000,
    random_seed=RANDOM_SEED
):
    """
    Estimate contrast-specific mediation models:

        Promotion Condition → Excitement → Outcome

    Planned contrasts (same order as H1):
        1. Chance-based vs. gamified certain
        2. Gamified certain vs. non-gamified certain

    Returns:
        effects_table
        indirect_effect_table

    Notes:
        - Non-gamified certain is the dummy-code reference category.
        - The chance-based vs. gamified-certain contrast is obtained by
          differencing the two dummy coefficients.
        - A fresh RNG with the common RANDOM_SEED is used in each call, so
          H3/H4 analyses use the same sequence of bootstrap row resamples.
    """

    analysis_df = data.copy()

    analysis_df["D_Game_Certain"] = (
        analysis_df["Condition"] == "Game_Certain"
    ).astype(int)
    analysis_df["D_Game_CB"] = (
        analysis_df["Condition"] == "Game_CB"
    ).astype(int)

    # a paths: promotion condition -> excitement
    mediator_model = smf.ols(
        "Excitement ~ D_Game_Certain + D_Game_CB",
        data=analysis_df
    ).fit()

    a_gc_vs_ng = mediator_model.params["D_Game_Certain"]
    a_cb_vs_gc = (
        mediator_model.params["D_Game_CB"]
        - mediator_model.params["D_Game_Certain"]
    )

    # Inferential tests for the planned a-path contrasts
    L_cb_vs_gc_a = np.array([0, -1, 1])
    L_gc_vs_ng_a = np.array([0, 1, 0])
    a_tests = {
        "Chance-based vs. gamified certain": mediator_model.t_test(L_cb_vs_gc_a),
        "Gamified certain vs. non-gamified": mediator_model.t_test(L_gc_vs_ng_a),
    }

    # Outcome model: b path and direct effects c'
    outcome_model = smf.ols(
        f"{outcome} ~ D_Game_Certain + D_Game_CB + Excitement",
        data=analysis_df
    ).fit()
    b_path = outcome_model.params["Excitement"]

    direct_gc_vs_ng = outcome_model.params["D_Game_Certain"]
    direct_cb_vs_gc = (
        outcome_model.params["D_Game_CB"]
        - outcome_model.params["D_Game_Certain"]
    )

    # Total-effect model: outcome ~ promotion condition
    total_model = smf.ols(
        f"{outcome} ~ D_Game_Certain + D_Game_CB",
        data=analysis_df
    ).fit()
    total_gc_vs_ng = total_model.params["D_Game_Certain"]
    total_cb_vs_gc = (
        total_model.params["D_Game_CB"]
        - total_model.params["D_Game_Certain"]
    )

    # Contrast tests for c' and c using the fitted covariance matrices
    # Parameter order: Intercept, D_Game_Certain, D_Game_CB [, Excitement]
    L_cb_vs_gc_direct = np.array([0, -1, 1, 0])
    L_gc_vs_ng_direct = np.array([0, 1, 0, 0])
    L_cb_vs_gc_total = np.array([0, -1, 1])
    L_gc_vs_ng_total = np.array([0, 1, 0])

    direct_tests = {
        "Chance-based vs. gamified certain": outcome_model.t_test(L_cb_vs_gc_direct),
        "Gamified certain vs. non-gamified": outcome_model.t_test(L_gc_vs_ng_direct),
    }
    total_tests = {
        "Chance-based vs. gamified certain": total_model.t_test(L_cb_vs_gc_total),
        "Gamified certain vs. non-gamified": total_model.t_test(L_gc_vs_ng_total),
    }

    contrasts = {
        "Chance-based vs. gamified certain": {
            "a": a_cb_vs_gc, "c_prime": direct_cb_vs_gc, "c": total_cb_vs_gc
        },
        "Gamified certain vs. non-gamified": {
            "a": a_gc_vs_ng, "c_prime": direct_gc_vs_ng, "c": total_gc_vs_ng
        },
    }

    # Nonparametric bootstrap of a*b
    rng = np.random.default_rng(random_seed)
    n = len(analysis_df)
    boot_effects = {name: [] for name in contrasts}

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot = analysis_df.iloc[idx].copy()

        X_m = np.column_stack([
            np.ones(n),
            boot["D_Game_Certain"].to_numpy(),
            boot["D_Game_CB"].to_numpy()
        ])
        y_m = boot["Excitement"].to_numpy()
        coef_m = np.linalg.lstsq(X_m, y_m, rcond=None)[0]

        boot_a_gc_vs_ng = coef_m[1]
        boot_a_cb_vs_gc = coef_m[2] - coef_m[1]

        X_y = np.column_stack([
            np.ones(n),
            boot["D_Game_Certain"].to_numpy(),
            boot["D_Game_CB"].to_numpy(),
            boot["Excitement"].to_numpy()
        ])
        y_y = boot[outcome].to_numpy()
        coef_y = np.linalg.lstsq(X_y, y_y, rcond=None)[0]
        boot_b = coef_y[3]

        boot_effects["Chance-based vs. gamified certain"].append(
            boot_a_cb_vs_gc * boot_b
        )
        boot_effects["Gamified certain vs. non-gamified"].append(
            boot_a_gc_vs_ng * boot_b
        )

    effects_rows = []
    indirect_rows = []
    for contrast, vals in contrasts.items():
        estimate = vals["a"] * b_path
        values = np.asarray(boot_effects[contrast])
        ci_low, ci_high = np.percentile(values, [2.5, 97.5])

        atest = a_tests[contrast]
        dtest = direct_tests[contrast]
        ttest = total_tests[contrast]
        a_p = float(np.asarray(atest.pvalue).squeeze())
        b_p = float(outcome_model.pvalues["Excitement"])
        d_p = float(np.asarray(dtest.pvalue).squeeze())
        t_p = float(np.asarray(ttest.pvalue).squeeze())

        effects_rows.append({
            "Contrast": contrast,
            "a": vals["a"],
            "p (a)": "< .001" if a_p < .001 else f"{a_p:.3f}",
            "b": b_path,
            "p (b)": "< .001" if b_p < .001 else f"{b_p:.3f}",
            "Direct Effect c'": vals["c_prime"],
            "p (c')": "< .001" if d_p < .001 else f"{d_p:.3f}",
            "Total Effect c": vals["c"],
            "p (c)": "< .001" if t_p < .001 else f"{t_p:.3f}",
        })

        indirect_rows.append({
            "Contrast": contrast,
            "Indirect Effect (a×b)": estimate,
            "Boot LLCI": ci_low,
            "Boot ULCI": ci_high,
            # Directional hypothesis: support requires the entire CI > 0.
            "Indirect Effect Supported": "Yes" if ci_low > 0 else "No"
        })

    effects_table = pd.DataFrame(effects_rows).round({
        "a": 3, "b": 3, "Direct Effect c'": 3, "Total Effect c": 3
    })
    indirect_effect_table = pd.DataFrame(indirect_rows).round({
        "Indirect Effect (a×b)": 3, "Boot LLCI": 3, "Boot ULCI": 3
    })

    return effects_table, indirect_effect_table


print("✓ Reusable mediation function defined successfully.")


### 4.4.2 Mediation for Urge to Buy (H3a)

Estimates are reported separately for the two planned contrasts. The mediator model uses all three conditions, so its standard errors can differ from those of the pairwise H1 tests.

In [ ]:
# H3a: Promotion Condition → Excitement → Urge to Buy

h3a_effects, h3a_indirect = run_mediation(
    data=df,
    outcome="Urge_to_Buy",
    n_boot=10_000
)

print("✓ H3a mediation analysis completed successfully.\n")
print("Contrast-specific paths and effects")
display(h3a_effects)
print("\nBootstrapped indirect effects")
display(h3a_indirect)


### 4.4.3 Mediation for Purchase Intention (H3b)

H3b is evaluated using the same two planned contrasts and bootstrap procedure as H3a. The *a*-path estimate for chance-based vs. gamified certain is again identical to H1a, while inferential statistics for the mediator-model coefficient are based on the full three-condition regression.


In [ ]:
# H3b: Promotion Condition → Excitement → Purchase Intention

h3b_effects, h3b_indirect = run_mediation(
    data=df,
    outcome="Purchase_Intention",
    n_boot=10_000
)

print("✓ H3b mediation analysis completed successfully.\n")
print("Contrast-specific paths and effects")
display(h3b_effects)
print("\nBootstrapped indirect effects")
display(h3b_indirect)


## 4.5 Moderated Mediation Analysis (H4)

H4 tests whether the indirect effect of promotion condition on the consumer outcomes through excitement depends on IBT, consistent with Hayes' (2022) PROCESS Model 14 logic. The analysis (1) estimates outcome regressions containing promotion condition, excitement, IBT, and excitement × IBT; (2) estimates conditional indirect effects at low (−1 SD), mean, and high (+1 SD) IBT using nonparametric bootstrapping; and (3) evaluates the index of moderated mediation.


### 4.5.1 Reusable Moderated Mediation Function

H4a and H4b use the same second-stage moderated mediation procedure and differ only in the outcome variable. The function below mean-centers excitement and IBT, estimates the moderated outcome model, and uses 10,000 bootstrap resamples for the conditional indirect effects and index of moderated mediation.


In [ ]:
def run_moderated_mediation(
    data,
    outcome,
    n_boot=10_000,
    random_seed=RANDOM_SEED
):
    """
    Estimate a second-stage moderated mediation model:

    Promotion Condition → Excitement → Outcome
                                  ↑
                                 IBT

    Returns:
        regression_table
        model_fit_table
        conditional_indirect_table
        index_table
    """

    # --------------------------------------------------------
    # Prepare local analysis data
    # --------------------------------------------------------

    analysis_df = data.copy()

    # Condition indicators
    # Reference: Non-gamified certain
    analysis_df["D_Game_Certain"] = (
        analysis_df["Condition"] == "Game_Certain"
    ).astype(int)

    analysis_df["D_Game_CB"] = (
        analysis_df["Condition"] == "Game_CB"
    ).astype(int)

    # Center mediator and moderator
    excitement_mean = analysis_df["Excitement"].mean()
    ibt_mean = analysis_df["IBT"].mean()
    ibt_sd = analysis_df["IBT"].std(ddof=1)

    analysis_df["Excitement_c"] = (
        analysis_df["Excitement"] - excitement_mean
    )

    analysis_df["IBT_c"] = (
        analysis_df["IBT"] - ibt_mean
    )

    analysis_df["Excitement_x_IBT"] = (
        analysis_df["Excitement_c"]
        * analysis_df["IBT_c"]
    )

    # ========================================================
    # 1. Mediator model
    # ========================================================

    mediator_model = smf.ols(
        "Excitement ~ D_Game_Certain + D_Game_CB",
        data=analysis_df
    ).fit()

    # Planned a-path contrasts
    a_gc_vs_ng = (
        mediator_model.params["D_Game_Certain"]
    )

    a_cb_vs_gc = (
        mediator_model.params["D_Game_CB"]
        - mediator_model.params["D_Game_Certain"]
    )

    # ========================================================
    # 2. Moderated outcome regression
    # ========================================================

    outcome_model = smf.ols(
        f"""
        {outcome} ~
        D_Game_Certain +
        D_Game_CB +
        Excitement_c +
        IBT_c +
        Excitement_x_IBT
        """,
        data=analysis_df
    ).fit()

    predictors = [
        "D_Game_Certain",
        "D_Game_CB",
        "Excitement_c",
        "IBT_c",
        "Excitement_x_IBT"
    ]

    predictor_labels = {
        "D_Game_Certain":
            "Gamified certain vs. non-gamified",
        "D_Game_CB":
            "Gamified chance-based vs. non-gamified",
        "Excitement_c":
            "Excitement",
        "IBT_c":
            "Impulse Buying Tendency",
        "Excitement_x_IBT":
            "Excitement × IBT"
    }

    regression_results = []

    for predictor in predictors:

        ci_low, ci_high = (
            outcome_model.conf_int()
            .loc[predictor]
        )

        p_value = outcome_model.pvalues[predictor]

        regression_results.append({
            "Predictor": predictor_labels[predictor],
            "B": outcome_model.params[predictor],
            "SE": outcome_model.bse[predictor],
            "t": outcome_model.tvalues[predictor],
            "p": (
                "< .001"
                if p_value < .001
                else f"{p_value:.3f}"
            ),
            "95% CI":
                f"[{ci_low:.2f}, {ci_high:.2f}]"
        })

    regression_table = pd.DataFrame(
        regression_results
    )

    regression_table = regression_table.round({
        "B": 3,
        "SE": 3,
        "t": 3
    })

    model_fit_table = pd.DataFrame({
        "Statistic": [
            "R²",
            "Adjusted R²"
        ],
        "Value": [
            outcome_model.rsquared,
            outcome_model.rsquared_adj
        ]
    }).round(3)

    # Outcome-model coefficients needed for indirect effects
    b_excitement = (
        outcome_model.params["Excitement_c"]
    )

    b_interaction = (
        outcome_model.params["Excitement_x_IBT"]
    )

    # Moderator levels
    ibt_levels = {
        "Low (-1 SD)": -ibt_sd,
        "Mean": 0.0,
        "High (+1 SD)": ibt_sd
    }

    # ========================================================
    # 3. Bootstrap
    # ========================================================

    rng = np.random.default_rng(random_seed)
    n = len(analysis_df)

    boot_indirect = {
        ("Chance-based vs. gamified certain", level): []
        for level in ibt_levels
    }

    boot_indirect.update({
        ("Gamified certain vs. non-gamified", level): []
        for level in ibt_levels
    })

    boot_index_cb_vs_gc = []
    boot_index_gc_vs_ng = []

    for _ in range(n_boot):

        sample_idx = rng.integers(
            0,
            n,
            size=n
        )

        boot = (
            analysis_df
            .iloc[sample_idx]
            .copy()
        )

        # ----------------------------------------------------
        # Bootstrap mediator model
        # ----------------------------------------------------

        X_m = np.column_stack([
            np.ones(len(boot)),
            boot["D_Game_Certain"].to_numpy(),
            boot["D_Game_CB"].to_numpy()
        ])

        y_m = boot["Excitement"].to_numpy()

        coef_m = np.linalg.lstsq(
            X_m,
            y_m,
            rcond=None
        )[0]

        boot_a_gc_vs_ng = coef_m[1]

        boot_a_cb_vs_gc = (
            coef_m[2] - coef_m[1]
        )

        # ----------------------------------------------------
        # Bootstrap outcome model
        # ----------------------------------------------------

        excitement_c = (
            boot["Excitement"].to_numpy()
            - excitement_mean
        )

        ibt_c = (
            boot["IBT"].to_numpy()
            - ibt_mean
        )

        interaction = (
            excitement_c * ibt_c
        )

        X_y = np.column_stack([
            np.ones(len(boot)),
            boot["D_Game_Certain"].to_numpy(),
            boot["D_Game_CB"].to_numpy(),
            excitement_c,
            ibt_c,
            interaction
        ])

        y_y = boot[outcome].to_numpy()

        coef_y = np.linalg.lstsq(
            X_y,
            y_y,
            rcond=None
        )[0]

        boot_b_excitement = coef_y[3]
        boot_b_interaction = coef_y[5]

        # ----------------------------------------------------
        # Conditional indirect effects
        # ----------------------------------------------------

        for level_name, w in ibt_levels.items():

            conditional_b = (
                boot_b_excitement
                + boot_b_interaction * w
            )

            boot_indirect[
                (
                    "Chance-based vs. gamified certain",
                    level_name
                )
            ].append(
                boot_a_cb_vs_gc * conditional_b
            )

            boot_indirect[
                (
                    "Gamified certain vs. non-gamified",
                    level_name
                )
            ].append(
                boot_a_gc_vs_ng * conditional_b
            )

        # ----------------------------------------------------
        # Indices of moderated mediation
        # ----------------------------------------------------

        boot_index_cb_vs_gc.append(
            boot_a_cb_vs_gc
            * boot_b_interaction
        )

        boot_index_gc_vs_ng.append(
            boot_a_gc_vs_ng
            * boot_b_interaction
        )

    # ========================================================
    # 4. Conditional indirect-effect table
    # ========================================================

    conditional_results = []

    for contrast, a_path in [
        (
            "Chance-based vs. gamified certain",
            a_cb_vs_gc
        ),
        (
            "Gamified certain vs. non-gamified",
            a_gc_vs_ng
        )
    ]:

        for level_name, w in ibt_levels.items():

            conditional_b = (
                b_excitement
                + b_interaction * w
            )

            indirect_effect = (
                a_path * conditional_b
            )

            boot_values = np.asarray(
                boot_indirect[
                    (contrast, level_name)
                ]
            )

            ci_low, ci_high = np.percentile(
                boot_values,
                [2.5, 97.5]
            )

            conditional_results.append({
                "Contrast": contrast,
                "IBT Level": level_name,
                "Indirect Effect": indirect_effect,
                "Boot LLCI": ci_low,
                "Boot ULCI": ci_high,
                "Boot CI Excludes Zero": (
                    "Yes"
                    if (ci_low > 0 or ci_high < 0)
                    else "No"
                )
            })

    conditional_indirect_table = pd.DataFrame(
        conditional_results
    ).round({
        "Indirect Effect": 3,
        "Boot LLCI": 3,
        "Boot ULCI": 3
    })

    # ========================================================
    # 5. Index of moderated mediation
    # ========================================================

    index_results = []

    index_specs = [
        (
            "Gamified certain vs. non-gamified",
            a_gc_vs_ng * b_interaction,
            boot_index_gc_vs_ng
        ),
        (
            "Chance-based vs. gamified certain",
            a_cb_vs_gc * b_interaction,
            boot_index_cb_vs_gc
        )
    ]

    for contrast, estimate, distribution in index_specs:

        ci_low, ci_high = np.percentile(
            distribution,
            [2.5, 97.5]
        )

        index_results.append({
            "Contrast": contrast,
            "Index": estimate,
            "Boot LLCI": ci_low,
            "Boot ULCI": ci_high,
            "Moderated Mediation": (
                "Yes"
                if (ci_low > 0 or ci_high < 0)
                else "No"
            )
        })

    index_table = pd.DataFrame(
        index_results
    ).round({
        "Index": 3,
        "Boot LLCI": 3,
        "Boot ULCI": 3
    })

    return (
        regression_table,
        model_fit_table,
        conditional_indirect_table,
        index_table
    )


print("✓ Reusable moderated mediation function defined successfully.")


### 4.5.2 Moderated Mediation for Urge to Buy (H4a)

H4a proposes that the indirect effect of promotion condition on urge to buy through excitement is stronger at higher IBT. The analysis reports the moderated outcome regression, bootstrapped conditional indirect effects, and index of moderated mediation.


#### 4.5.2.1 Moderated Outcome Regression

Urge to buy is regressed on promotion condition, excitement, IBT, and excitement × IBT. The non-gamified certain condition is the reference category. Excitement and IBT are mean-centered before constructing the interaction to facilitate interpretation and reduce multicollinearity. The interaction provides an initial moderation test; the final H4a evaluation uses the bootstrapped conditional indirect effects and index of moderated mediation.


In [ ]:
# H4a: run the common moderated mediation procedure
(
    h4a_results,
    h4a_model_fit,
    h4a_conditional_indirect,
    h4a_index
) = run_moderated_mediation(
    data=df,
    outcome="Urge_to_Buy",
    n_boot=10_000
)

print("✓ H4a moderated mediation analysis completed successfully.")
display(h4a_results)
print("\nModel fit")
display(h4a_model_fit)


#### 4.5.2.2 Bootstrapped Conditional Indirect Effects

Conditional indirect effects are estimated with 10,000 nonparametric bootstrap resamples at low (−1 SD), mean, and high (+1 SD) IBT. The two planned contrasts are gamified chance-based vs. gamified certain and gamified certain vs. non-gamified certain. Bootstrap confidence intervals indicate whether each indirect effect differs from zero.


In [ ]:
# H4a: bootstrapped conditional indirect effects
display(h4a_conditional_indirect)


#### 4.5.2.3 Index of Moderated Mediation

The index of moderated mediation quantifies whether the indirect effect on urge to buy through excitement changes with IBT. It is estimated with 10,000 bootstrap resamples; moderated mediation is supported when the bootstrap confidence interval excludes zero.


In [ ]:
# H4a: index of moderated mediation
display(h4a_index)


### 4.5.3 Moderated Mediation for Purchase Intention (H4b)

H4b proposes that the indirect effect of promotion condition on purchase intention through excitement is stronger at higher IBT. The same reusable procedure is applied with purchase intention as the outcome.


In [ ]:
# H4b: Run Moderated Mediation Analysis

(
    h4b_regression,
    h4b_model_fit,
    h4b_indirect,
    h4b_index
) = run_moderated_mediation(
    data=df,
    outcome="Purchase_Intention",
    n_boot=10_000
)

print("✓ H4b moderated mediation analysis completed successfully.")


#### 4.5.3.1 Moderated Outcome Regression

Purchase intention is regressed on promotion condition, excitement, IBT, and excitement × IBT. The non-gamified certain condition is the reference category, and excitement and IBT are mean-centered before constructing the interaction. The interaction provides an initial moderation test; the final H4b evaluation uses the bootstrapped conditional indirect effects and index of moderated mediation.


In [ ]:
# H4b: Moderated Outcome Regression

display(h4b_regression)

print("\nModel fit")

display(h4b_model_fit)


#### 4.5.3.2 Bootstrapped Conditional Indirect Effects

Conditional indirect effects are estimated with 10,000 nonparametric bootstrap resamples at low (−1 SD), mean, and high (+1 SD) IBT for the two planned promotion contrasts. Bootstrap confidence intervals indicate whether each indirect effect differs from zero.


In [ ]:
# H4b: Bootstrapped Conditional Indirect Effects

display(h4b_indirect)


#### 4.5.3.3 Index of Moderated Mediation

The index of moderated mediation tests whether the indirect effect on purchase intention through excitement changes with IBT. It is estimated with 10,000 bootstrap resamples; moderated mediation is supported when the bootstrap confidence interval excludes zero.


In [ ]:
# H4b: Index of Moderated Mediation

display(h4b_index)


# 5. Robustness Analyses

Additional regressions test whether the main relationships remain stable after controlling for perceived discount value, product involvement, perceived annoyance, and demographic characteristics. Robustness is supported when the substantive conclusions remain unchanged after including these controls.


In [ ]:
# Reusable Robustness Regression Function

def run_robustness_regression(
    data,
    outcome,
    controls
):
    """
    Runs an OLS regression with excitement and additional controls.

    Parameters
    ----------
    data : DataFrame
    outcome : str
    controls : list

    Returns
    -------
    DataFrame
    """

    predictors = ["Excitement"] + controls

    formula = (
        outcome +
        " ~ " +
        " + ".join(predictors)
    )

    model = smf.ols(
        formula,
        data=data
    ).fit()

    results = []

    for predictor in predictors:

        ci_low, ci_high = (
            model.conf_int()
            .loc[predictor]
        )

        p = model.pvalues[predictor]

        results.append({

            "Predictor": predictor,

            "B": model.params[predictor],

            "SE": model.bse[predictor],

            "t": model.tvalues[predictor],

            "p": (
                "< .001"
                if p < .001
                else f"{p:.3f}"
            ),

            "95% CI":
                f"[{ci_low:.2f}, {ci_high:.2f}]"

        })

    table = pd.DataFrame(results)

    table = table.round({

        "B":3,

        "SE":3,

        "t":3

    })

    model_fit = pd.DataFrame({

        "Statistic":[
            "R²",
            "Adjusted R²"
        ],

        "Value":[
            model.rsquared,
            model.rsquared_adj
        ]

    }).round(3)

    return table, model_fit

print("✓ Robustness regression function defined successfully.")


## 5.1 Controlling for Perceived Discount Value

This analysis tests whether the findings remain stable after controlling for perceived discount value, thereby assessing whether perceived economic value accounts for the observed relationships.


In [ ]:
# Robustness Check: Perceived Discount Value

# Urge to Buy
discount_urge, _ = run_robustness_regression(
    data=df,
    outcome="Urge_to_Buy",
    controls=["Discount_Value"]
)

# Purchase Intention
discount_purchase, _ = run_robustness_regression(
    data=df,
    outcome="Purchase_Intention",
    controls=["Discount_Value"]
)

# Predictor labels

label_mapping = {
    "Discount_Value": "Perceived Discount Value"
}

discount_urge["Predictor"] = (
    discount_urge["Predictor"]
    .replace(label_mapping)
)

discount_purchase["Predictor"] = (
    discount_purchase["Predictor"]
    .replace(label_mapping)
)

# Display results

print("✓ Perceived discount value robustness check completed successfully.\n")

print("Urge to Buy")
display(discount_urge)

print("\nPurchase Intention")
display(discount_purchase)


## 5.2 Controlling for Product Involvement

This analysis controls for product involvement to assess whether the effects of excitement on urge to buy and purchase intention remain after accounting for individual differences in involvement with the product category.


In [ ]:
# Robustness Check: Product Involvement

# Urge to Buy
involvement_urge, _ = run_robustness_regression(
    data=df,
    outcome="Urge_to_Buy",
    controls=["Product_Involvement"]
)

# Purchase Intention
involvement_purchase, _ = run_robustness_regression(
    data=df,
    outcome="Purchase_Intention",
    controls=["Product_Involvement"]
)

# Thesis-friendly predictor labels

label_mapping = {
    "Product_Involvement": "Product Involvement"
}

involvement_urge["Predictor"] = (
    involvement_urge["Predictor"]
    .replace(label_mapping)
)

involvement_purchase["Predictor"] = (
    involvement_purchase["Predictor"]
    .replace(label_mapping)
)

# Display results

print("✓ Product involvement robustness check completed successfully.\n")

print("Urge to Buy")
display(involvement_urge)

print("\nPurchase Intention")
display(involvement_purchase)


## 5.3 Controlling for Annoyance

This analysis controls for perceived annoyance to assess whether the effects of excitement remain after accounting for potential frustration elicited by the promotion.


In [ ]:
# Robustness Check: Perceived Annoyance

# Urge to Buy
annoyance_urge, _ = run_robustness_regression(
    data=df,
    outcome="Urge_to_Buy",
    controls=["Annoyance"]
)

# Purchase Intention
annoyance_purchase, _ = run_robustness_regression(
    data=df,
    outcome="Purchase_Intention",
    controls=["Annoyance"]
)

# Predictor labels

label_mapping = {
    "Annoyance": "Perceived Annoyance"
}

annoyance_urge["Predictor"] = (
    annoyance_urge["Predictor"]
    .replace(label_mapping)
)

annoyance_purchase["Predictor"] = (
    annoyance_purchase["Predictor"]
    .replace(label_mapping)
)

# Display results

print("✓ Perceived annoyance robustness check completed successfully.\n")

print("Urge to Buy")
display(annoyance_urge)

print("\nPurchase Intention")
display(annoyance_purchase)


## 5.4 Controlling for Demographic Variables

This analysis controls for age as a continuous variable and gender as a categorical variable. Robustness is supported if the effect of excitement remains substantively unchanged after accounting for these demographics.


In [ ]:
# Reusable Demographic Robustness Function

def run_demographic_robustness(data, outcome):

    model = smf.ols(
        f"{outcome} ~ Excitement + Age + C(Gender)",
        data=data
    ).fit()

    results = []

    for predictor in model.params.index:

        if predictor == "Intercept":
            continue

        ci_low, ci_high = model.conf_int().loc[predictor]
        p = model.pvalues[predictor]

        predictor_labels = {
            "Excitement": "Excitement",
            "Age": "Age",
            "C(Gender)[T.2]": "Gender (Female)",
            "C(Gender)[T.4]": "Gender (Other)"
        }
        label = predictor_labels.get(predictor, predictor)

        results.append({

            "Predictor": label,

            "B": round(model.params[predictor], 3),

            "SE": round(model.bse[predictor], 3),

            "t": round(model.tvalues[predictor], 3),

            "p": (
                "< .001"
                if p < .001
                else f"{p:.3f}"
            ),

            "95% CI":
                f"[{ci_low:.2f}, {ci_high:.2f}]"

        })

    table = pd.DataFrame(results)

    # ----------------------------------------------------
    # Consistent predictor order
    # ----------------------------------------------------

    order = [
        "Excitement",
        "Age",
        "Gender (Female)",
        "Gender (Other)"
    ]

    table["Predictor"] = pd.Categorical(
        table["Predictor"],
        categories=order,
        ordered=True
    )

    table = (
        table
        .sort_values("Predictor")
        .reset_index(drop=True)
    )

    return table


print("✓ Demographic robustness function defined successfully.")


In [ ]:
# Robustness Check: Demographic Variables

demo_urge = run_demographic_robustness(
    data=df,
    outcome="Urge_to_Buy"
)

demo_purchase = run_demographic_robustness(
    data=df,
    outcome="Purchase_Intention"
)

print("✓ Demographic robustness check completed successfully.\n")

print("Urge to Buy")
display(demo_urge)

print("\nPurchase Intention")
display(demo_purchase)


## 5.5 Chapter Summary

The robustness analyses assess the main relationships after controlling for perceived discount value, product involvement, perceived annoyance, age, and gender. These models complement rather than replace the primary hypothesis tests.


# 6. Export of Results

The analysis tables are exported to a single Excel workbook in the `outputs` directory defined in Section 0.3.


In [ ]:
# Reuse the output directory defined in Section 0.3
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Output directory ready: {OUTPUT_DIR.resolve()}")


In [ ]:
# Export All Tables to Excel

output_file = OUTPUT_DIR / "Thesis_Tables.xlsx"

# Tables to export

tables = {

    # ========================================================
    # Chapter 3
    # ========================================================

    "Reliability": reliability_table,
    "Descriptive Statistics": descriptive_table,
    "Correlation Matrix": correlation_table,

    "Manipulation Descriptives": presentation_table,
    "Manipulation ANOVA": anova_table,
    "Manipulation Post Hoc": posthoc_table,
    "Manipulation Effect Sizes": effect_size_table,

    "Age Summary": age_summary,
    "Gender Distribution": gender_table,
    "Condition Distribution": condition_table,

    # ========================================================
    # Chapter 4 – H1
    # ========================================================

    "H1 Omnibus ANOVA": anova_summary,
    "H1 Planned Comparisons": planned_comparisons,
    "H1 Effect Sizes": effect_table,

    # ========================================================
    # Chapter 4 – H2
    # ========================================================

    "H2a Urge to Buy": h2a_table,
    "H2b Purchase Intention": h2b_table,

    # ========================================================
    # Chapter 4 – H3: Mediation
    # ========================================================

    "H3a Paths and Effects": h3a_effects,
    "H3a Indirect Effects": h3a_indirect,
    "H3b Paths and Effects": h3b_effects,
    "H3b Indirect Effects": h3b_indirect,

    # ========================================================
    # Chapter 4 – H4: Moderated Mediation
    # ========================================================

    "H4a Regression": h4a_results,
    "H4a Model Fit": h4a_model_fit,
    "H4a Conditional Indirect": h4a_conditional_indirect,
    "H4a Index": h4a_index,
    "H4b Regression": h4b_regression,
    "H4b Model Fit": h4b_model_fit,
    "H4b Conditional Indirect": h4b_indirect,
    "H4b Index": h4b_index,

    # ========================================================
    # Robustness Checks
    # ========================================================

    "Discount Urge": discount_urge,
    "Discount Purchase": discount_purchase,

    "Involvement Urge": involvement_urge,
    "Involvement Purchase": involvement_purchase,

    "Annoyance Urge": annoyance_urge,
    "Annoyance Purchase": annoyance_purchase,

    "Demographics Urge": demo_urge,
    "Demographics Purchase": demo_purchase
}

# Export workbook

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    for sheet_name, table in tables.items():

        table.to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False
        )

print(
    f"✓ Successfully exported {len(tables)} tables.\n"
    f"Location: {output_file.resolve()}"
)


## Notebook Completed

The notebook provides a reproducible workflow from raw-data screening through variable construction, scale evaluation, hypothesis testing, robustness analyses, and result export.
